## Import Modules

In [1]:
import random
from pathlib import Path

import numpy as np

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Prepare Datas

In [ ]:
# setting the random seed
def set_seed(seed=42):
    random.seed(seed)       # the internal module `random`
    np.random.seed(seed)    # for NumPy
    torch.manual_seed(seed) # for torch

set_seed(42)

In [3]:
# using `torch.cuda.is_available()` to check the current environment
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
# using `sklearn.load_digits()` to load dataset(handwritten scripts of 8*8, labels are 0-9)
digits=load_digits()

# Normalization
X=digits.data.astype(np.float32)/16.0   # the origin digit ranges from 0 to 16

y=digits.target.astype(np.int64)

print("feature shape:",X.shape)
print("label shape:",y.shape)
print("label range:",np.unique(y))

feature shape: (1797, 64)
label shape: (1797,)
label range: [0 1 2 3 4 5 6 7 8 9]


In [5]:
# splitting the training datasets and testing datasets using `sklearn.model_selection.train_test_split()`
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,      # 20% for the testing datasets
    random_state=42,    # set random state for this splitting
    stratify=y,         # Important : try to make the label consistent in training datasets and testing datasets
)

print("training feature shape:",X_train.shape)
print("testing feature shape:",X_test.shape)
print("training label shape:",y_train.shape)
print("testing label shape:",y_test.shape)

training feature shape: (1437, 64)
testing feature shape: (360, 64)
training label shape: (1437,)
testing label shape: (360,)


In [8]:
# using `torch.from_numpy()` to transfer NumPy array to PyTorch Tensor
train_dataset=TensorDataset(    #TensorDataset(features,labels)
    torch.from_numpy(X_train),
    torch.from_numpy(y_train),
)
test_dataset=TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

print("number of training data:",len(train_dataset))
print("number of test data:",len(test_dataset))

number of training data: 1437
number of test data: 360


Instead of using `sklearn`, we can use `TensorDataset`

```python
from torch.utils.data import TensorDataset, random_split

digits = load_digits()

#loading data
X = digits.data.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

#create Dataset
dataset = TensorDataset(
    torch.from_numpy(X),
    torch.from_numpy(y)
)

#splitting
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
```

In [11]:
# loading dataset with `DataLoader`
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader=DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

# checking one batch
sample_features,sample_labels=next(iter(train_loader))

print("shape of feature in batch:",sample_features.shape)   #output: torch.Size([batch_size, feature size])
print("shape of label in batch:",sample_labels.shape)

shape of feature in batch: torch.Size([64, 64])
shape of label in batch: torch.Size([64])


## Define MLP

The structure of the network `64 features → Linear(64, 128) → ReLU → Linear(128, 64) → ReLU → Linear(64, 10)`

- `__init__()` defines what layers does the network contain

- `forward()` defines how input data going through these layers

<img src="insert_images/MLP_neural_network_and_weights_EN.png" alt="Batch Epoch" width="1200">

In [12]:
class DigitMLP(nn.Module):
    '''input 64 image features, and output 10 numerical categories scores '''
    def __init__(self):
        super().__init__()

        self.network=nn.Sequential(
            nn.Linear(64,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)    # output logits
        )

    def forward(self,x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
model=DigitMLP().to(device) #object.to() is the method of PyTorch 
#  define a instant `model`
print(model)    #checkout the network structure

DigitMLP(
  (network): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


## Config Loss Function(CrossEntropyLoss) and Optmizer(Adam)

In [15]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
# `model.parameters()` returns "iterator/generator", and the element inside is `torch.nn.parameter.Parameter`(a subclass of Tensor, and `requires_grad=True`)
# equivalent form: `[p for p in model.parameters()]`

print("Loss Function:", loss_fn)
print("optimizer:", optimizer.__class__.__name__)

Loss Function: CrossEntropyLoss()
optimizer: Adam


## Evaluate Function

In [21]:
# using `evaluate()` to calculate the average loss and accuracy of test data 
def evaluate(model: nn.Module,loader: DataLoader,loss_fn: nn.Module,device: torch.device):
    model.eval()

    total_loss=0.0
    correct=0
    total=0

    with torch.no_grad():
         for features, labels in loader:
              features=features.to(device)
              labels=labels.to(device)

              logits=model(features)    #logits.shape [64,10] #labels.shape [64,]
              loss=loss_fn(logits,labels)    #Important : loss.shape torch.Size([])  Because in `nn.CorssEntropyLoss()`, we default set `reduction=mean`!
                              # for example: torch.tensor(3.14) torch.Size([]) it's a scalar   Because `reduction=mean` returns scalar
                                           # torch.tensor([3.14]) torch.Size([1]) it's a vector
              batch_size=labels.size(0) #equvalent to `batch_size=labels.size(dim=0)`
              total_loss+=loss.item()*batch_size
              correct+=(logits.argmax(dim=1)==labels).sum().item()
              total+=batch_size

    average_loss=total_loss/total
    accuracy=correct/total

    return average_loss, accuracy

## Training model

Training model is the core part of the project

ervery batch will execute the operation:

1. `optimizer.zero_grad()`: clear up the gradients of the late iteration.

2. `logits=model(feature)`: forward propagation, get the original output.

3. `loss=loss_fn(logits,labels)`: calculate the loss.

4. `loss.backward()`: backpropagation for calculating the gradients.

5. `optimizer.step()`: upgrade the parameters according to gradients.

In [23]:
epochs=30

for epoch in range(1,epochs+1):
    model.train()   #`train()` is the method of `torch.nn.Module`

    train_loss_sum=0.0
    train_correct=0
    train_total=0

    for features, labels in train_loader:
        features=features.to(device)
        labels=labels.to(device)

        # 1
        optimizer.zero_grad()
        # 2
        logits=model(features)
        # 3
        loss=loss_fn(logits,labels)
        # 4
        loss.backward()
        # 5
        optimizer.step()

        #calculate the loss and correct number in this epoch
        batch_size=labels.size(0)
        train_loss_sum+=loss.item()*batch_size
        train_correct+=(logits.argmax(dim=1)==labels).sum().item()
        train_total += batch_size

    # check the performance of training dataset and test dataset regularly
    if epoch==1 or epoch%5==0 or epoch==epochs:
        train_loss=train_loss_sum/train_total
        train_acc=train_correct/train_total

        test_loss, test_acc=evaluate(model,test_loader,loss_fn,device)

        print(f"Epoch {epoch}/{epochs} |" f"train loss: {train_acc:.4f} |"
              f"test loss: {test_loss:.2%} |" f"test acc: {test_acc:.2f}")

Epoch 1/30 |train loss: 0.9903 |test loss: 8.11% |test acc: 0.98
Epoch 5/30 |train loss: 0.9951 |test loss: 7.93% |test acc: 0.98
Epoch 10/30 |train loss: 0.9986 |test loss: 7.23% |test acc: 0.98
Epoch 15/30 |train loss: 0.9986 |test loss: 7.24% |test acc: 0.98
Epoch 20/30 |train loss: 0.9979 |test loss: 7.31% |test acc: 0.98
Epoch 25/30 |train loss: 0.9986 |test loss: 7.00% |test acc: 0.98
Epoch 30/30 |train loss: 0.9993 |test loss: 7.00% |test acc: 0.98


## Check the prediction of model

In [ ]:
model.eval()

sample_x,sample_y=next(iter(test_loader))

with torch.no_grad():
    predictions=model(sample_x.to(device)).argmax(dim=1).cpu()

for true_label, pred in zip(sample_y[:12].tolist(), predictions[:12].tolist()):
    mark = "✓" if true_label == pred else "✗"
    print(f"  {true_label} -> {pred} {mark}")

  5 -> 5 ✓
  2 -> 2 ✓
  8 -> 8 ✓
  1 -> 1 ✓
  7 -> 7 ✓
  2 -> 2 ✓
  6 -> 6 ✓
  2 -> 2 ✓
  6 -> 6 ✓
  5 -> 5 ✓
  0 -> 0 ✓
  5 -> 5 ✓


## Save the trained model parameters

In [28]:
# using `model.state_dict()` to get the model parameters, and using `torch.save()` to save to digit_mlp.pth
# note: digit_mlp.pth only saves the parameters, not the model structure. We need to redefine the netwoek structure
model_path=Path("digit_mlp.pth")

torch.save(model.state_dict(),model_path)